# 02 — Demand Forecasting Model

Builds a demand model that predicts `quantity_sold` from price, competitor price, and seasonality. This is the **predict-then-optimize bridge** between ML and OR.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_prep import load_processed
from demand_model import DemandModel, build_demand_models

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Processed Data

In [ ]:
weekly_df, elasticity_df = load_processed()
print(f"Data shape: {weekly_df.shape}")
weekly_df.head()

## 2. Train Global Demand Model

In [ ]:
model = build_demand_models(weekly_df, model_type='xgboost')

## 3. Model Evaluation

In [ ]:
print("Model Metrics:")
for k, v in model.metrics.items():
    if k != 'feature_importance':
        print(f"  {k}: {v}")

if 'feature_importance' in model.metrics:
    fi = model.metrics['feature_importance']
    plt.figure(figsize=(8, 4))
    plt.barh(list(fi.keys()), list(fi.values()))
    plt.title('Feature Importance')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 4. Demand Curves for Sample SKUs

In [ ]:
top_skus = weekly_df.groupby('product_id')['quantity_sold'].sum().nlargest(4).index
locations = weekly_df['location'].unique()[:2]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, sku in enumerate(top_skus):
    ax = axes[idx // 2, idx % 2]
    sku_data = weekly_df[weekly_df['product_id'] == sku]
    mean_comp = sku_data['competitor_price'].mean() if 'competitor_price' in sku_data.columns else 50
    
    for loc in locations:
        prices, demands = model.get_demand_curve(
            competitor_price=mean_comp,
            week=26,
            category=sku_data['category'].iloc[0] if 'category' in sku_data.columns else 'default',
            location=loc,
            price_range=(sku_data['price'].min() * 0.5, sku_data['price'].max() * 1.5),
            n_points=30
        )
        ax.plot(prices, demands, label=loc, linewidth=2)
    
    ax.set_title(f'{sku}')
    ax.set_xlabel('Price ($)')
    ax.set_ylabel('Expected Demand')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Forecasted Demand Curves', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Predicted vs Actual (Time-based Split)

In [ ]:
from sklearn.preprocessing import LabelEncoder

test_df = weekly_df.sort_values('week').reset_index(drop=True)
split_idx = int(len(test_df) * 0.8)
test_split = test_df.iloc[split_idx:].copy()

test_split['predicted'] = model.predict(test_split)

plt.figure(figsize=(8, 8))
plt.scatter(test_split['quantity_sold'], test_split['predicted'], alpha=0.4, s=15)
max_val = max(test_split['quantity_sold'].max(), test_split['predicted'].max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Quantity Sold')
plt.ylabel('Predicted Quantity Sold')
plt.title('Predicted vs Actual (Test Set)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Save Model for Optimization Layer

The trained model is now ready to be consumed by the MILP and GA solvers as `d_i(p_i)`.

In [ ]:
model.save()
print("Demand model saved. Ready for optimization layer.")